In [0]:
products = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_products_master")
enriched_orders = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_orders_enriched")

print(f"✅ Loaded products: {products.count():,}")
print(f"✅ Loaded orders: {enriched_orders.count():,}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# PRODUCT PERFORMANCE METRICS
# ====================================================================
print("📊 Calculating product performance metrics...\n")

# Add performance rankings
window_revenue = Window.orderBy(col("total_revenue").desc())
window_units = Window.orderBy(col("total_units_sold").desc())
window_category = Window.partitionBy("product_category_english").orderBy(col("total_revenue").desc())

product_performance = (products
    .select(
        "product_id",
        "product_category_name",
        "product_category_english",
        "product_weight_g",
        "product_volume_cm3",
        "total_units_sold",
        "total_orders",
        "unique_customers",
        "total_revenue",
        "avg_price",
        "avg_freight_cost",
        "avg_revenue_per_unit",
        "avg_review_score",
        "total_reviews",
        "avg_delivery_days",
        "late_delivery_count",
        "late_delivery_rate",
        "unique_sellers",
        "first_sale_date",
        "last_sale_date",
        "days_since_last_sale",
        "product_lifecycle_days",
        "product_performance",
        "is_highly_rated"
    )
    # Add revenue rank (overall)
    .withColumn(
        "revenue_rank",
        rank().over(window_revenue)
    )
    
    # Add units sold rank (overall)
    .withColumn(
        "units_rank",
        rank().over(window_units)
    )
    
    # Add category rank
    .withColumn(
        "category_rank",
        rank().over(window_category)
    )
    
    # Calculate profit margin estimate (simplified)
    .withColumn(
        "estimated_profit_margin",
        round(((col("avg_price") - col("avg_freight_cost")) / col("avg_price")) * 100, 2)
    )
    
    # Flag star products (top 10% by revenue)
    .withColumn(
        "is_star_product",
        when(col("revenue_rank") <= products.count() * 0.10, True).otherwise(False)
    )
    
    # Product health score (0-100)
    .withColumn(
        "product_health_score",
        round(
            (col("avg_review_score") / 5.0 * 40) +  # 40% weight on reviews
            (when(col("days_since_last_sale") <= 30, 30)  # 30% weight on recency
             .when(col("days_since_last_sale") <= 90, 20)
             .when(col("days_since_last_sale") <= 180, 10)
             .otherwise(0)) +
            (when(col("late_delivery_rate") <= 10, 30)  # 30% weight on delivery
             .when(col("late_delivery_rate") <= 20, 20)
             .when(col("late_delivery_rate") <= 30, 10)
             .otherwise(0)),
            0
        )
    )
    
    # Recommended action
    .withColumn(
        "recommended_action",
        when(
            (col("is_star_product") == True) & (col("avg_review_score") >= 4.0),
            "Promote heavily, ensure stock"
        )
        .when(
            (col("total_units_sold") >= 50) & (col("avg_review_score") < 3.0),
            "Investigate quality issues"
        )
        .when(
            col("days_since_last_sale") >= 180,
            "Consider discontinuing"
        )
        .when(
            (col("total_units_sold") < 10) & (col("product_lifecycle_days") >= 180),
            "Low performer - review or discount"
        )
        .when(
            col("late_delivery_rate") >= 30,
            "Improve logistics/seller performance"
        )
        .otherwise("Monitor performance")
    )
    
    .withColumn("_created_at", current_timestamp())
)

print(f"✅ Product performance calculated: {product_performance.count():,} products")
print("=" * 80 + "\n")

In [0]:
product_perf_table = f"{CATALOG}.{GOLD_SCHEMA}.gold_product_performance"

(product_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(product_perf_table))

print(f"✅ Written: {product_perf_table}")
print(f"   Rows: {product_performance.count():,}")
print(f"   Columns: {len(product_performance.columns)}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# CATEGORY PERFORMANCE SUMMARY
# ====================================================================
print("📊 Creating category performance summary...\n")

category_performance = (product_performance
    .groupBy("product_category_english")
    .agg(
        count("product_id").alias("total_products"),
        sum("total_units_sold").alias("category_units_sold"),
        sum("total_revenue").alias("category_revenue"),
        avg("avg_price").alias("avg_category_price"),
        avg("avg_review_score").alias("avg_category_review"),
        avg("late_delivery_rate").alias("avg_late_delivery_rate"),
        sum(when(col("is_star_product") == True, 1).otherwise(0)).alias("star_products_count")
    )
    .withColumn(
        "revenue_percentage",
        round((col("category_revenue") / sum("category_revenue").over(Window.partitionBy())) * 100, 2)
    )
    .withColumn("_created_at", current_timestamp())
    .orderBy(col("category_revenue").desc())
)

category_perf_table = f"{CATALOG}.{GOLD_SCHEMA}.gold_category_performance"

(category_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(category_perf_table))

print(f"✅ Written: {category_perf_table}")
print("=" * 80 + "\n")

In [0]:
%sql
-- Top 20 products by revenue
SELECT 
    product_id,
    product_category_english,
    total_units_sold,
    ROUND(total_revenue, 2) as revenue,
    ROUND(avg_price, 2) as avg_price,
    ROUND(avg_review_score, 2) as avg_review,
    revenue_rank,
    product_health_score,
    recommended_action
FROM workspace.retail_gold.gold_product_performance
ORDER BY revenue_rank
LIMIT 20;

In [0]:
%sql
-- Worst performing products
SELECT 
    product_id,
    product_category_english,
    total_units_sold,
    ROUND(total_revenue, 2) as revenue,
    days_since_last_sale,
    product_lifecycle_days,
    ROUND(avg_review_score, 2) as avg_review,
    product_health_score,
    recommended_action
FROM workspace.retail_gold.gold_product_performance
WHERE recommended_action IN ('Consider discontinuing', 'Low performer - review or discount')
ORDER BY product_health_score ASC
LIMIT 20;

In [0]:
%sql
-- Category performance overview
SELECT 
    product_category_english,
    total_products,
    category_units_sold,
    ROUND(category_revenue, 2) as revenue,
    revenue_percentage,
    ROUND(avg_category_price, 2) as avg_price,
    ROUND(avg_category_review, 2) as avg_review,
    star_products_count
FROM workspace.retail_gold.gold_category_performance
ORDER BY category_revenue DESC
LIMIT 20;

In [0]:
%sql
-- Star products (top 10% by revenue)
SELECT 
    product_category_english,
    COUNT(*) as star_product_count,
    SUM(total_units_sold) as total_units,
    ROUND(SUM(total_revenue), 2) as total_revenue,
    ROUND(AVG(avg_review_score), 2) as avg_review
FROM workspace.retail_gold.gold_product_performance
WHERE is_star_product = true
GROUP BY product_category_english
ORDER BY total_revenue DESC;

In [0]:
print("\n" + "=" * 80)
print("✅ PRODUCT PERFORMANCE ANALYSIS COMPLETE")
print("=" * 80)
print("\nCreated tables:")
print(f"  - {CATALOG}.{GOLD_SCHEMA}.gold_product_performance")
print(f"  - {CATALOG}.{GOLD_SCHEMA}.gold_category_performance")

# Show quick stats
star_count = product_performance.filter(col("is_star_product") == True).count()
total_products = product_performance.count()

print(f"\n📊 Quick Stats:")
print(f"  - Total Products: {total_products:,}")
print(f"  - Star Products: {star_count:,} ({star_count/total_products*100:.1f}%)")

print("\n" + "=" * 80)
print("🎉 PHASE 3 - GOLD LAYER COMPLETE!")
print("=" * 80)
print("\n✅ All Gold Tables Created:")
print("  - gold_revenue_daily")
print("  - gold_revenue_monthly")
print("  - gold_revenue_overall")
print("  - gold_customer_cohorts")
print("  - gold_customer_retention")
print("  - gold_rfm_scores")
print("  - gold_rfm_segment_summary")
print("  - gold_product_performance")
print("  - gold_category_performance")
print("\n📝 Next Phase: ML Models (churn prediction, demand forecasting)")
print("=" * 80 + "\n")
